# Desafio 2 — Ciência de Dados 1

**Aluno(a): Leonardo Xerez**

Rode cada célula e **escreva sua resposta** no campo indicado. Você **não** precisa
escrever código — o objetivo é interpretar e comparar os resultados. Os arquivos
`.csv` estão nesta pasta.

> **Regra deste desafio:** toda afirmação que menciona um valor deve **trazer o valor**,
> copiado da saída da sua célula. Os números mudam de caderno para caderno.


In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from sklearn.model_selection import (train_test_split, cross_val_score, KFold,
                                     StratifiedKFold, GroupKFold, TimeSeriesSplit)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

def modelo():
    """O modelo usado nas questoes de base simples."""
    return Pipeline([("esc", StandardScaler()),
                     ("clf", LogisticRegression(max_iter=1000))])

def floresta():
    """O modelo usado nas questoes de painel e de serie."""
    return RandomForestClassifier(n_estimators=60, random_state=0)

print("ambiente pronto")

## Exercício 1 — Um corte só é jogar a moeda

A base `q1_amostra.csv` tem 300 clientes, 8 variáveis e um alvo binário. O código treina o MESMO modelo dez vezes, mudando apenas a **semente do sorteio** que separa treino e teste (70/30).

**(a)** Dê a **menor** e a **maior** nota entre as dez, e a **amplitude** (maior − menor).

**(b)** Se você tivesse rodado só a semente 0 e parado ali, que número teria anunciado? Por que anunciar esse número é um problema?

**(c)** Cada uma das dez medições é **honesta** — o teste nunca foi usado no treino. Então o problema do corte único é de **viés** ou de **instabilidade**? Justifique.

In [ ]:
df = pd.read_csv("q1_amostra.csv")
X = df.drop(columns="alvo").values
y = df["alvo"].values

notas = []
for s in range(10):
    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=0.3, random_state=s, stratify=y)
    nota = modelo().fit(Xtr, ytr).score(Xte, yte)
    notas.append(nota)
    print(f"semente {s}: {nota:.4f}")

notas = np.array(notas)
print(f"\nmenor {notas.min():.4f} | maior {notas.max():.4f}"
      f" | amplitude {notas.max() - notas.min():.4f}")

**Resposta:**

_(escreva aqui)_

## Exercício 2 — O mesmo grupo dos dois lados

A base `q5_painel.csv` tem **40 empresas acompanhadas por 16 trimestres** — 640 linhas, mas só 40 empresas distintas. O alvo diz se a empresa superou o setor naquele trimestre. O código compara o rodízio aleatório com o `GroupKFold`, que mantém cada empresa inteira de um lado só.

**(a)** Dê as **duas notas** e a **queda** entre elas.

**(b)** Explique o mecanismo: o que o sorteio aleatório faz com as 16 linhas de uma mesma empresa, e o que o modelo passa a fazer por causa disso?

**(c)** Qual das duas notas você reportaria para "este modelo vai ser usado em empresas que ainda não existem na base"? E a outra nota — ela responde a que pergunta?

In [ ]:
df = pd.read_csv("q5_painel.csv")
cols = ["porte", "margem", "crescimento", "endividamento", "liquidez"]
X = df[cols].values
y = df["superou_setor"].values
g = df["id_empresa"].values
print("linhas:", len(df), "| empresas distintas:", df.id_empresa.nunique(),
      "| trimestres:", df.periodo.nunique())

aleat = cross_val_score(floresta(), X, y,
                        cv=KFold(5, shuffle=True, random_state=0)).mean()
grupo = cross_val_score(floresta(), X, y, cv=GroupKFold(5), groups=g).mean()
print(f"\nrodizio ALEATORIO : {aleat:.4f}")
print(f"GroupKFold        : {grupo:.4f}")
print(f"queda             : {aleat - grupo:.4f}")

**Resposta:**

_(escreva aqui)_

## Exercício 3 — k-fold: a média e o desvio

Mesma base da questão do corte único. Agora, em vez de um corte, o rodízio: cada pedaço é teste uma vez e treino nas outras. O código roda com **k = 5** e com **k = 10**.

**(a)** Dê a **média** e o **desvio-padrão** de cada um dos dois (k = 5 e k = 10).

**(b)** Compare o **desvio-padrão do k-fold** com a **amplitude do hold-out** da outra questão, citando os dois números. O que a comparação mostra?

**(c)** O desvio-padrão que a validação cruzada devolve mede o quê, exatamente? E o que muda, na prática, ao passar de k = 5 para k = 10?

In [ ]:
df = pd.read_csv("q1_amostra.csv")
X = df.drop(columns="alvo").values
y = df["alvo"].values

for k in [5, 10]:
    cv = StratifiedKFold(k, shuffle=True, random_state=0)
    notas = cross_val_score(modelo(), X, y, cv=cv)
    print(f"k={k:2d} | notas: {np.round(notas, 3)}")
    print(f"       media {notas.mean():.4f} +/- {notas.std():.4f}\n")

**Resposta:**

_(escreva aqui)_

## Exercício 4 — O pré-processamento que vaza

A base `q3_muitas_colunas.csv` tem **120 linhas e 180 colunas**. O código faz a mesma coisa de duas maneiras: na primeira, escolhe as 15 melhores colunas olhando a base inteira e **só depois** valida; na segunda, a escolha das colunas entra no `Pipeline` e é refeita dentro de cada dobra.

**(a)** Dê as **duas notas** e a **diferença** entre elas.

**(b)** A primeira nota é uma mentira. Qual é o mecanismo — o que exatamente o `SelectKBest` enxergou que não devia?

**(c)** Rode a última célula, que revela como o alvo desta base foi construído. À luz disso, qual é o desempenho **real** possível aqui — e o que o `Pipeline` faz de diferente dentro da validação cruzada?

In [ ]:
df = pd.read_csv("q3_muitas_colunas.csv")
X = df.drop(columns="alvo").values
y = df["alvo"].values
print("formato:", X.shape, "  <- 120 linhas, 180 colunas")

cv = StratifiedKFold(5, shuffle=True, random_state=0)

# (1) ERRADO: escolhe as colunas vendo o y de TODAS as linhas
sel = SelectKBest(f_classif, k=15).fit(X, y)
errado = cross_val_score(LogisticRegression(max_iter=1000),
                         sel.transform(X), y, cv=cv).mean()

# (2) CERTO: a escolha entra no Pipeline e e refeita em cada dobra
certo = cross_val_score(
    Pipeline([("sel", SelectKBest(f_classif, k=15)),
              ("clf", LogisticRegression(max_iter=1000))]), X, y, cv=cv).mean()

print(f"\n(1) selecao ANTES da validacao : {errado:.4f}")
print(f"(2) selecao DENTRO do Pipeline : {certo:.4f}")
print(f"    diferenca                  : {errado - certo:.4f}")

In [ ]:
# Como o alvo desta base foi construido:
print("proporcao de 1s:", round(y.mean(), 3))
print("correlacao media |r| entre as colunas e o alvo:",
      round(float(np.abs(np.corrcoef(X.T, y)[-1, :-1]).mean()), 4))
print("\nO alvo foi SORTEADO NO CARA OU COROA, sem relacao nenhuma")
print("com as 180 colunas. Nao existe padrao nenhum para aprender.")

**Resposta:**

_(escreva aqui)_

## Exercício 5 — Quando a classe é rara demais para sortear

A base `q4_raros.csv` tem 200 linhas e pouquíssimos positivos. O código mostra quantos positivos caem em cada dobra com o `KFold` comum e com o `StratifiedKFold`, e o F1 de cada dobra nos dois casos.

**(a)** Quantos positivos a base tem, e quantos caíram em cada dobra com o `KFold` comum?

**(b)** O que acontece com o F1 na dobra que ficou sem nenhum positivo — e por quê?

**(c)** Com o `StratifiedKFold`, o que mudou na distribuição dos positivos e no F1?

**(d)** Qual é o **maior k** que faz sentido nesta base? Dê o número e justifique.

In [ ]:
df = pd.read_csv("q4_raros.csv")
X = df.drop(columns="alvo").values
y = df["alvo"].values
print("linhas:", len(y), "| positivos:", int(y.sum()),
      f"({100*y.mean():.1f}%)")

for nome, cv in [("KFold comum      ", KFold(5, shuffle=True, random_state=0)),
                 ("StratifiedKFold  ", StratifiedKFold(5, shuffle=True, random_state=0))]:
    partes = list(cv.split(X, y))
    pos = [int(y[te].sum()) for _, te in partes]
    f1 = cross_val_score(modelo(), X, y, cv=partes, scoring="f1")
    print(f"\n{nome} positivos por dobra: {pos}")
    print(f"{' '*19}F1 por dobra       : {np.round(f1, 3)}")
    print(f"{' '*19}F1 medio           : {f1.mean():.4f}")

**Resposta:**

_(escreva aqui)_

## Exercício 6 — O futuro dentro do treino

A base `q6_serie.csv` é uma série de 600 trimestres **em ordem**. O código compara o rodízio aleatório com o `TimeSeriesSplit`, que sempre treina no passado e testa no futuro. A última célula mostra como a relação entre os indicadores e o alvo se comporta ao longo da série.

**(a)** Dê as **duas notas** e a **queda** entre elas.

**(b)** Por que sortear as linhas coloca o futuro dentro do treino — e, olhando a última célula, por que nesta série isso pesa tanto?

**(c)** A base da questão das empresas também tem uma coluna de período. Usar o `GroupKFold` lá resolveu também o problema do tempo? Justifique.

In [ ]:
df = pd.read_csv("q6_serie.csv")          # ja vem em ordem de trimestre
cols = ["trimestre", "ind_a", "ind_b", "ind_c", "ind_d"]
X = df[cols].values
y = df["alvo"].values
print("trimestres:", df.trimestre.min(), "a", df.trimestre.max())

aleat = cross_val_score(floresta(), X, y,
                        cv=KFold(5, shuffle=True, random_state=0)).mean()
tempo = cross_val_score(floresta(), X, y, cv=TimeSeriesSplit(5)).mean()
print(f"\nrodizio ALEATORIO : {aleat:.4f}")
print(f"TimeSeriesSplit   : {tempo:.4f}")
print(f"queda             : {aleat - tempo:.4f}")

In [ ]:
# A relacao entre ind_a e o alvo, na primeira e na segunda metade da serie:
meio = len(df) // 2
for rot, parte in [("1a metade", df.iloc[:meio]), ("2a metade", df.iloc[meio:])]:
    r = np.corrcoef(parte["ind_a"], parte["alvo"])[0, 1]
    print(f"{rot}: correlacao entre ind_a e o alvo = {r:+.3f}")

**Resposta:**

_(escreva aqui)_